# Recommendation RS — Protocollo 10/20 e doppia valutazione A/B

**Notebook autonomo** · Non richiede né modifica `src/`, MCP, Qdrant o workflow LLM.

**Scopo.** Preparare il confronto *new-user cold start* su MovieLens usando il catalogo TMDb già arricchito:

- **Train/validation/test disgiunti per utenti**: utenti valutati mai visti da Matrix Factorization durante il training globale.
- Per ogni utente eleggibile (almeno 30 rating **nel catalogo**), **10 rating di profilo**, **20 rating fissi per ground truth A** e **tutti i rimanenti rating come holdout aggiuntivo**.
- Livelli `k = 0, 1, 3, 5, 10`: si rivelano i primi `k` rating della stessa sequenza ordinata. Tutti gli algoritmi ricevono i medesimi input e producono ranking degli stessi candidati.
- **A (principale):** positivi tra i 20 rating di test fissi. **B (robustezza):** positivi tra *tutti* i rating dell'utente non ancora rivelati (inclusi profilo non rivelato + holdout aggiuntivo).
- I film già rivelati sono **esclusi** dai candidati. Quelli non rivelati restano raccomandabili. I film senza voto sono *non osservati*, non necessariamente sgraditi.

**Non** addestra modelli né calcola i ranking MF/GTE: prepara dataset e offre la funzione di valutazione per ranking futuri. Include un auto-test senza modello e produce file versionati, separando rigorosamente gli input pubblicabili ai recommender dalle etichette private usate solo per metriche/simulatore.

### Precondizioni e limiti

Dataset `datasets/raw/ml-latest-small/ratings.csv`, `movies.csv`, e catalogo `datasets/processed/dataset_enriched_movielens_tmdb.csv`. Il catalogo definisce l'universo candidate (nell'audit: **1.996 film**). Non riusare i suoi `ml_tags`, `ml_rating_mean` o altri attributi dipendenti dagli utenti test per preparare embedding o popularity: le feature GTE devono essere *split-safe* (idealmente soli metadata TMDb).

**Protocollo di prima esecuzione:** un solo split utenti 80/10/10, tre repliche casuali 10/20 per utente. La cross-validation multi-fold e il temporal holdout restano possibili estensioni, **non** sono eseguite qui. Le repliche dello stesso utente NON sono osservazioni indipendenti e dovranno essere aggregate per utente prima di inferenze statistiche.

## 0. Configurazione e percorsi

Posiziona questo notebook in `<project_root>/notebooks/`. Non richiede import da `src/movie_rag`. Cambia soltanto `PROJECT_ROOT_OVERRIDE` se il progetto usa un'altra struttura.

In [2]:
from pathlib import Path
import hashlib
import json
import math
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT_OVERRIDE = None   # es. Path('/Users/.../progetto')
PROTOCOL_ID = 'rs_coldstart_v1'
SEED = 20261021
N_PROFILE = 10
N_FIXED_GT = 20
LEVELS = (0, 1, 3, 5, 10)
N_REPETITIONS = 3
POS_THRESHOLD = 4.0
NEG_THRESHOLD = 2.0
TRAIN_FRACTION = 0.80
VAL_FRACTION = 0.10  # resto = test
K_METRICS = 10
SAVE_OUTPUTS = True


def find_root():
    if PROJECT_ROOT_OVERRIDE is not None:
        return Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
    current = Path.cwd().resolve()
    for base in [current, *list(current.parents)[:5]]:
        if (base/'datasets/raw/ml-latest-small/ratings.csv').is_file() and \
           (base/'datasets/processed/dataset_enriched_movielens_tmdb.csv').is_file():
            return base
    raise FileNotFoundError('Dataset non disponibili: imposta PROJECT_ROOT_OVERRIDE alla root del progetto.')

ROOT = find_root()
RATINGS_PATH = ROOT/'datasets/raw/ml-latest-small/ratings.csv'
MOVIES_PATH = ROOT/'datasets/raw/ml-latest-small/movies.csv'
CATALOG_PATH = ROOT/'datasets/processed/dataset_enriched_movielens_tmdb.csv'
OUT = ROOT/'datasets/processed/recommendation_protocol'/PROTOCOL_ID
assert 0 < N_PROFILE and 0 < N_FIXED_GT and 0 < TRAIN_FRACTION < 1
assert N_REPETITIONS >= 1 and list(LEVELS) == sorted(set(LEVELS))
assert LEVELS[0] == 0 and LEVELS[-1] == N_PROFILE
assert all(0 <= k <= N_PROFILE for k in LEVELS)
print('Root:', ROOT)
print('Output directory:', OUT)
print('Protocollo:', PROTOCOL_ID, '| seed:', SEED, '| ripetizioni:', N_REPETITIONS)

Root: /Users/danieleraveglia/Documents/uniMi/TESI/progetto
Output directory: /Users/danieleraveglia/Documents/uniMi/TESI/progetto/datasets/processed/recommendation_protocol/rs_coldstart_v1
Protocollo: rs_coldstart_v1 | seed: 20261021 | ripetizioni: 3


## 1. Caricamento e controlli sul catalogo

Usiamo **solo `movieId`** per il join. Il numero di film non è forzato a 2.000 e NON rigeneriamo un catalogo dalla popolarità. La funzione qui sotto non legge metadata utente del catalogo arricchito.

In [4]:
ratings_all = pd.read_csv(RATINGS_PATH)
movies = pd.read_csv(MOVIES_PATH)
catalog = pd.read_csv(CATALOG_PATH, usecols=['movieId'])
for name, df, required in [
    ('ratings.csv', ratings_all, ['userId', 'movieId', 'rating']),
    ('movies.csv', movies, ['movieId']),
    ('catalog', catalog, ['movieId']),
]:
    if any(col not in df for col in required):
        raise ValueError(f'{name}: colonne mancanti {set(required)-set(df.columns)}')
    df['movieId'] = pd.to_numeric(df['movieId'], errors='raise').astype('int64')
if catalog.movieId.duplicated().any():
    raise ValueError('Catalogo duplicato in movieId')
if ratings_all[['userId', 'movieId']].duplicated().any():
    raise ValueError('Rating duplicati (userId,movieId)')
ratings_all['userId'] = pd.to_numeric(ratings_all.userId, errors='raise').astype('int64')
ratings_all['rating'] = pd.to_numeric(ratings_all.rating, errors='raise').astype('float64')
assert ratings_all.rating.notna().all()
assert ratings_all.rating.between(0.5, 5.0).all()
assert np.isclose((ratings_all.rating*2)%1, 0).all()
assert set(catalog.movieId).issubset(set(movies.movieId))

ids = set(catalog.movieId)
ratings = ratings_all.loc[ratings_all.movieId.isin(ids), ['userId','movieId','rating']].copy()
if ratings.empty:
    raise ValueError('Nessuna sovrapposizione tra rating e catalogo')
ratings = ratings.sort_values(['userId','movieId']).reset_index(drop=True)
counts = ratings.groupby('userId').size().rename('n_ratings').reset_index()
counts['eligible'] = counts.n_ratings >= N_PROFILE+N_FIXED_GT
print('Catalogo:',len(ids),'film; rating:',len(ratings),'utenti:',len(counts))
print('Idonei >=30 rating:',int(counts.eligible.sum()))
display(counts.n_ratings.describe(percentiles=[.1,.25,.5,.75,.9]).to_frame('rating_per_user'))

Catalogo: 1996 film; rating: 78219 utenti: 610
Idonei >=30 rating: 480


,rating_per_user
count,610.000000
mean,128.227869
std,171.160932
min,7.000000
10%,22.000000
25%,32.000000
50%,63.000000
75%,142.750000
90%,318.100000
max,1505.000000


## 2. Split **per utenti** train/validation/test

Stratifichiamo soltanto sulla disponibilità di ≥30 rating, **non** sui rating positivi nascosti. Assicuriamo la presenza di utenti eleggibili sia in validazione sia in test. Gli utenti di test/validazione **non** contribuiscono a training globale di MF né alle statistiche popularity. Lo split è fisso e ricostruibile dalla seed.

In [6]:
def split_users(counts_df, seed):
    rng = np.random.default_rng(seed)
    parts = []
    for eligible, grp in counts_df.groupby('eligible', sort=True):
        user_ids = np.sort(grp.userId.to_numpy())
        rng.shuffle(user_ids)
        n = len(user_ids)
        n_val = int(round(n * VAL_FRACTION))
        n_test = int(round(n * (1 - TRAIN_FRACTION - VAL_FRACTION)))
        for i, user_id in enumerate(user_ids):
            split = 'validation' if i < n_val else ('test' if i < n_val + n_test else 'train')
            parts.append({'userId':int(user_id), 'split':split, 'eligible':bool(eligible),
                          'n_ratings':int(grp.loc[grp.userId==user_id,'n_ratings'].iloc[0])})
    result = pd.DataFrame(parts).sort_values('userId').reset_index(drop=True)
    assert result.userId.nunique() == len(counts_df)
    return result

users = split_users(counts, SEED)
print('Split per utenti:')
print('(False: utenti con meno di 30 rating, True: utenti idonei)')
display(pd.crosstab(users['split'], users['eligible'], margins=True))
train_users = set(users.loc[users.split=='train','userId'])
train_ratings = ratings[ratings.userId.isin(train_users)].copy().reset_index(drop=True)
assert not set(users.loc[users.split!='train','userId']).intersection(set(train_ratings.userId))
coverage = train_ratings.groupby('movieId').size().reindex(catalog.movieId, fill_value=0)
print('Rating di training:',len(train_ratings),'| film con almeno 1 rating train:',int((coverage>0).sum()), '/',len(catalog))
print('Film con almeno 5 rating train:',int((coverage>=5).sum()))

Split per utenti:
(False: utenti con meno di 30 rating, True: utenti idonei)


eligible,False,True,All
split,,,
test,13,48,61
train,104,384,488
validation,13,48,61
All,130,480,610


Rating di training: 62423 | film con almeno 1 rating train: 1996 / 1996
Film con almeno 5 rating train: 1996


## 3. Costruzione delle 3 repliche del protocollo 10/20

Per ogni utente eleggibile di **validazione o test**, ogni replica:

1. Campiona 30 **rating** uniformemente tra tutti i suoi rating sul catalogo, senza guardare la classe del voto.
2. Assegna i primi 10, dopo shuffle, al **profilo privato** con ordine `reveal_order=1..10`.
3. Assegna 20 al **ground truth fisso A**.
4. Classifica **tutti gli altri** come holdout aggiuntivo, utile SOLO alla robustezza B.

Nessun rating degli utenti non-train passa al training di MF. Le repliche possono riutilizzare uno stesso rating, ma MAI dentro una singola replica tra i tre insiemi. L'ordine di rivelazione (con rating positivi/intermedi/negativi) rimane identico ai diversi `k` della stessa replica.

Per testare il protocollo iniziale **non imponiamo** ≥3 positivi nei 20: riportiamo gli eventuali scenari senza positivi (metriche di ranking undefined/NaN), senza selezionare a posteriori repliche favorevoli. Eventuali criteri futuri di idoneità saranno decisi sul set di validazione, prima del test finale.

In [7]:
def make_scenarios(ratings_df, split_df, seed, repetitions):
    eligible = split_df[(split_df['eligible']) & (split_df['split'].isin(['validation','test']))]
    split_of = eligible.set_index('userId')['split'].to_dict()
    user_groups = {int(uid):grp[['movieId','rating']].sort_values('movieId').reset_index(drop=True)
                   for uid,grp in ratings_df.groupby('userId') if uid in split_of}
    rng = np.random.default_rng(seed)
    profile_rows, fixed_rows, additional_rows, manifest_rows = [], [], [], []
    for uid in sorted(user_groups):
        grp = user_groups[uid]
        assert len(grp) >= N_PROFILE + N_FIXED_GT
        for rep in range(repetitions):
            scenario_id = f'{split_of[uid]}_u{uid}_r{rep}'
            chosen = rng.choice(len(grp), size=N_PROFILE+N_FIXED_GT, replace=False)
            chosen = chosen[rng.permutation(len(chosen))]
            profile_idx = chosen[:N_PROFILE]
            fixed_idx = chosen[N_PROFILE:]
            extra_idx = np.setdiff1d(np.arange(len(grp)), chosen, assume_unique=True)
            for order, idx in enumerate(profile_idx, start=1):
                profile_rows.append({'scenario_id':scenario_id,'userId':uid,'movieId':int(grp.iloc[idx].movieId),
                                     'rating':float(grp.iloc[idx].rating),'reveal_order':order})
            for idx in fixed_idx:
                fixed_rows.append({'scenario_id':scenario_id,'userId':uid,'movieId':int(grp.iloc[idx].movieId),
                                   'rating':float(grp.iloc[idx].rating)})
            for idx in extra_idx:
                additional_rows.append({'scenario_id':scenario_id,'userId':uid,'movieId':int(grp.iloc[idx].movieId),
                                        'rating':float(grp.iloc[idx].rating)})
            manifest_rows.append({'scenario_id':scenario_id,'userId':uid,'split':split_of[uid],
                                  'rep':rep,'total_user_ratings':len(grp),'n_profile':N_PROFILE,
                                  'n_fixed_gt':N_FIXED_GT,'n_additional':len(extra_idx),
                                  'n_fixed_gt_pos':int((grp.iloc[fixed_idx].rating >= POS_THRESHOLD).sum())})
    profile = pd.DataFrame(profile_rows)
    fixed = pd.DataFrame(fixed_rows)
    additional = pd.DataFrame(additional_rows, columns=['scenario_id','userId','movieId','rating'])
    scenarios = pd.DataFrame(manifest_rows)
    if scenarios.empty:
        raise ValueError('Nessun utente valutabile. Verificare filtri e audit.')
    return scenarios, profile, fixed, additional

scenarios, private_profile, private_fixed, private_extra = make_scenarios(
    ratings, users, SEED+17, N_REPETITIONS
)
print('Scenari:')
display(scenarios.groupby('split').agg(n_scenarios=('scenario_id','size'),n_utenti=('userId','nunique'),
                                      target_positivi_medi=('n_fixed_gt_pos','mean'),
                                      additional_medi=('n_additional','mean')).round(2))
print('Quota scenari A con almeno 1/3/5 positivi:',
      {n:round((scenarios.n_fixed_gt_pos>=n).mean()*100,2) for n in [1,3,5]})

Scenari:


,n_scenarios,n_utenti,target_positivi_medi,additional_medi
split,,,,
test,144,48,11.65,107.17
validation,144,48,10.62,149.67


Quota scenari A con almeno 1/3/5 positivi: {1: np.float64(100.0), 3: np.float64(97.92), 5: np.float64(93.4)}


## 4. Input visibili dei modelli e controlli anti-leakage

Ogni riga di `visible_interactions.csv` contiene **esattamente un rating rivelato** allo specifico livello `k`. Con `k=0` non ci sono righe (occorre fare join contro `scenario_levels.csv`: assenza di righe significa profilo vuoto).

**Non consegnare ai modelli** `private_profile.csv`, `private_fixed_gt.csv` o `private_additional.csv`: rimangono separati, per simulatore/valutatore. Il simulatore LLM, quando sarà introdotto, potrà ricevere soltanto i 10 rating del profilo, MAI fixed/extra. Nel confronto MF/GTE statico, i modelli ricevono soltanto `visible_interactions.csv` filtrato per `scenario_id` e `k`.

I candidati sono sempre i film di `catalog_movie_ids.csv` **meno quelli rivelati**, non soltanto i film che hanno rating nel ground truth.

In [8]:
level_rows=[]
observed_rows=[]
for rec in scenarios.itertuples(index=False):
    p = private_profile.loc[private_profile.scenario_id == rec.scenario_id].sort_values('reveal_order')
    for k in LEVELS:
        level_rows.append({'scenario_id':rec.scenario_id,'userId':int(rec.userId),
                           'split':rec.split,'rep':int(rec.rep),'k':int(k)})
        if k:
            for r in p.head(k).itertuples(index=False):
                observed_rows.append({'scenario_id':rec.scenario_id,'k':int(k),
                                      'movieId':int(r.movieId),'rating':float(r.rating),
                                      'reveal_order':int(r.reveal_order)})
scenario_levels = pd.DataFrame(level_rows)
visible = pd.DataFrame(observed_rows, columns=['scenario_id','k','movieId','rating','reveal_order'])

# Verifiche strutturali sulla completa separazione: eseguite PRIMA di scrivere i file.
assert scenarios.scenario_id.is_unique
assert len(private_profile)==N_PROFILE*len(scenarios)
assert len(private_fixed)==N_FIXED_GT*len(scenarios)
assert len(scenario_levels)==len(LEVELS)*len(scenarios)
assert visible.groupby(['scenario_id','k']).size().eq(visible.groupby(['scenario_id','k']).first().index.get_level_values('k')).all()
for s in scenarios.scenario_id:
    p=set(private_profile.loc[private_profile.scenario_id==s,'movieId'])
    f=set(private_fixed.loc[private_fixed.scenario_id==s,'movieId'])
    e=set(private_extra.loc[private_extra.scenario_id==s,'movieId'])
    assert len(p)==N_PROFILE and len(f)==N_FIXED_GT
    assert not(p&f or p&e or f&e)
    uid=int(scenarios.loc[scenarios.scenario_id==s,'userId'].iloc[0])
    assert p|f|e == set(ratings.loc[ratings.userId==uid,'movieId'])
assert all(set(visible.loc[(visible.scenario_id==s)&(visible.k==k),'movieId'])==
           set(private_profile.loc[(private_profile.scenario_id==s)&(private_profile.reveal_order<=k),'movieId'])
           for s in scenarios.scenario_id for k in LEVELS)
assert not set(train_ratings.userId)&set(scenarios.userId)
assert set(private_profile.movieId).issubset(ids)
assert set(private_fixed.movieId).issubset(ids)
assert set(private_extra.movieId).issubset(ids)
print('Verifiche PASS: split utenti disgiunti, profili/target/additional non sovrapposti, input progressivi coerenti.')
display(scenario_levels.groupby(['split','k']).size().unstack('k'))

Verifiche PASS: split utenti disgiunti, profili/target/additional non sovrapposti, input progressivi coerenti.


k,0,1,3,5,10
split,,,,,
test,144,144,144,144,144
validation,144,144,144,144,144


## 5. Esportazione separata: `inputs` e `private_evaluation`

sotto `/datasets/processed/recommendation_protocol/rs_coldstart_v1/`

I file qui prodotti rappresentano un **protocollo versionato**. Sotto `inputs/` ci sono soltanto il catalogo, i rating degli utenti di training, gli ID degli scenari e le interazioni rivelate. Non contengono la numerosità nascosta dello storico degli utenti di test. Sotto `private_evaluation/` ci sono le etichette nascoste, utilizzabili **solo** dal valutatore / simulatore. I file generati possono contenere ID MovieLens e rating: non condividerli con LLM esterni indiscriminatamente.

Non verrà creato né modificato alcun file dentro `src/movie_rag` o `notebooks` oltre al nuovo notebook salvato dall'utente. Se una cartella di output con questo ID contiene già risultati, il notebook si ferma invece di sovrascrivere silenziosamente le simulazioni.

In [9]:
def file_hash(path):
    digest=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(2**20),b''):
            digest.update(chunk)
    return digest.hexdigest()

write_plan={
    'inputs/catalog_movie_ids.csv':catalog[['movieId']].sort_values('movieId'),
    'private_evaluation/user_partitions.csv':users,
    'inputs/train_ratings.csv':train_ratings,
    'inputs/scenarios.csv':scenarios[['scenario_id','userId','split','rep']],  # nessun conteggio privato
    'private_evaluation/scenario_metadata.csv':scenarios,
    'inputs/scenario_levels.csv':scenario_levels,
    'inputs/visible_interactions.csv':visible,
    'private_evaluation/private_profile.csv':private_profile,
    'private_evaluation/private_fixed_gt.csv':private_fixed,
    'private_evaluation/private_additional.csv':private_extra,
    'private_evaluation/scenario_target_counts.csv':scenarios[['scenario_id','n_fixed_gt_pos']],
}
metadata={
    'protocol_id':PROTOCOL_ID,'seed':SEED,'profile_size':N_PROFILE,'fixed_holdout_size':N_FIXED_GT,
    'levels':list(LEVELS),'repetitions':N_REPETITIONS,'positive_rating_min':POS_THRESHOLD,
    'negative_rating_max':NEG_THRESHOLD,'item_universe_size':len(catalog),
    'training_n_users':len(train_users),'training_n_ratings':len(train_ratings),
    'validation_n_scenarios':int((scenarios.split=='validation').sum()),
    'test_n_scenarios':int((scenarios.split=='test').sum()),
    'sources':{'ratings_sha256':file_hash(RATINGS_PATH),'catalog_sha256':file_hash(CATALOG_PATH)},
    'note':'A: 20 fixed; B: all unraveled ratings; missing MovieLens rating = unjudged, not dislike. '
           'No MF or GTE training is performed here.',
}
if SAVE_OUTPUTS:
    # Evita export legacy di statistiche private accidentalmente lasciati in inputs/.
    stale=OUT/'inputs/user_partitions.csv'
    if stale.exists():
        raise RuntimeError(f'File legacy con informazioni private presente: {stale}. '
                           'Verifica e sposta/rimuovi tale file prima di rieseguire.')
    manifest_file=OUT/'protocol_manifest.json'
    if manifest_file.exists():
        old=json.loads(manifest_file.read_text(encoding='utf-8'))
        if old != metadata:
            raise FileExistsError(f'Output esistente con manifest diverso: {manifest_file}. '
                                  'Usa un altro PROTOCOL_ID; non sovrascrivere.')
        print('Manifest uguale già presente: riscrittura deterministica degli stessi dati.')
    for relative, df in write_plan.items():
        dest=OUT/relative
        dest.parent.mkdir(parents=True,exist_ok=True)
        df.to_csv(dest,index=False)
    manifest_file.write_text(json.dumps(metadata,indent=2,ensure_ascii=False),encoding='utf-8')
    print('Creati',len(write_plan),'CSV e protocol_manifest.json in',OUT)
else:
    print('SAVE_OUTPUTS=False: solo preparazione in memoria')

Creati 11 CSV e protocol_manifest.json in /Users/danieleraveglia/Documents/uniMi/TESI/progetto/datasets/processed/recommendation_protocol/rs_coldstart_v1


## 6. Valutatore A/B: stessi ranking, due insiemi di giudizi

**Contratto per i futuri algoritmi MF, GTE, LLM**: un CSV con le colonne:

`scenario_id, k, model, movieId, rank` (facoltativo `score`).

Il ranking può contenere solo la **top-K** (K≥10 per @10); i film non devono essere già rivelati. La funzione non rilancia i modelli: effettua il join con i rating **privati** del protocollo, e calcola entrambe le metriche A/B per lo stesso ranking.

- **A:** positivo se movieId nei **20 fixed holdout** e rating≥4.
- **B:** positivo se movieId in **tutti** i rating dell'utente non ancora rivelati (fixed+extra+profilo non rivelato) e rating≥4.
- I film con rating non noto hanno **rilevanza osservata = 0** per il calcolo standard delle metriche; NON significa che non piacciano all'utente.
- Un ranking che propone un item già rivelato è invalidato, non "corretto" silenziosamente.
- NDCG@10 e Recall@10 sono `NaN` se non ci sono target positivi osservati; HR@10 è anch'esso `NaN` in quel caso. Riferire sempre il conteggio degli scenari definiti.

**Nota metodologica:** la metrica B ha un numero di positivi potenziali dipendente da `k`. Per un confronto scientificamente corretto, confrontare i modelli **all'interno dello stesso schema e dello stesso k**, non i valori assoluti di A contro B.

In [10]:
def evaluation_truth(scenario_id, k, scheme):
    if scheme not in ('A','B'):
        raise ValueError('scheme deve essere A o B')
    if (scenario_id, k) not in valid_level_pairs:
        raise KeyError(f'Scenario/level non definito: {scenario_id}, {k}')
    f=private_fixed.loc[private_fixed.scenario_id==scenario_id,['movieId','rating']]
    if scheme=='A':
        return dict(zip(f.movieId.astype(int),f.rating.astype(float)))
    e=private_extra.loc[private_extra.scenario_id==scenario_id,['movieId','rating']]
    p=private_profile.loc[(private_profile.scenario_id==scenario_id)&
                          (private_profile.reveal_order>k), ['movieId','rating']]
    all_hidden=pd.concat([f,e,p],ignore_index=True)
    assert all_hidden.movieId.is_unique
    return dict(zip(all_hidden.movieId.astype(int),all_hidden.rating.astype(float)))

valid_level_pairs=set(zip(scenario_levels.scenario_id,scenario_levels.k))
observed_lookup={(s,int(k)):set(g.movieId.astype(int)) for (s,k),g in visible.groupby(['scenario_id','k'])}

def evaluate_rankings(rankings, top_k=K_METRICS):
    required={'scenario_id','k','model','movieId','rank'}
    missing=required-set(rankings.columns)
    if missing:
        raise ValueError(f'Mancano colonne ranking: {missing}')
    if rankings.empty:
        raise ValueError('Rankings vuoti')
    tmp=rankings.copy()
    if tmp[list(required)].isna().any().any():
        raise ValueError('Valori null in campi obbligatori dei ranking')
    tmp['k']=tmp['k'].astype(int)
    tmp['movieId']=tmp['movieId'].astype(int)
    tmp['rank']=tmp['rank'].astype(int)
    if top_k<=0:raise ValueError('top_k deve essere positivo')
    out=[]
    for (sid,k,model),g in tmp.groupby(['scenario_id','k','model'],sort=True):
        if (sid,k) not in valid_level_pairs:
            raise ValueError(f'Scenario non esistente: {(sid,k)}')
        if g.movieId.duplicated().any() or g['rank'].duplicated().any():
            raise ValueError(f'Duplicati nel ranking {(sid,k,model)}')
        if not set(g.movieId).issubset(ids):
            raise ValueError(f'Film fuori catalogo in {(sid,k,model)}')
        if set(g.movieId) & observed_lookup.get((sid,k),set()):
            raise ValueError(f'Il ranking suggerisce un film GIA rivelato: {(sid,k,model)}')
        g=g.sort_values('rank')
        if g['rank'].iloc[0]!=1 or not np.array_equal(g['rank'].to_numpy(),np.arange(1,len(g)+1)):
            raise ValueError(f'Rank non consecutivi da 1: {(sid,k,model)}')
        if len(g)<top_k:
            raise ValueError(f'Meno di {top_k} candidati nel ranking {(sid,k,model)}')
        recommendation_ids=g.head(top_k).movieId.astype(int).to_numpy()
        for scheme in ('A','B'):
            truth=evaluation_truth(sid,k,scheme)
            relevant={movie for movie,rating in truth.items() if rating >= POS_THRESHOLD}
            labels=np.array([1.0 if movie in relevant else 0.0 for movie in recommendation_ids],dtype=float)
            n_pos=len(relevant)
            if n_pos:
                dcg=float((labels/np.log2(np.arange(2,top_k+2))).sum())
                idcg=float((np.ones(min(top_k,n_pos))/np.log2(np.arange(2,min(top_k,n_pos)+2))).sum())
                ndcg=dcg/idcg
                recall=float(labels.sum()/n_pos)
                hr=float(labels.sum()>0)
            else:
                ndcg=recall=hr=np.nan
            out.append({'scenario_id':sid,'k':k,'model':model,'scheme':scheme,
                        'n_relevant':n_pos,'n_hits_topk':int(labels.sum()),
                        f'ndcg@{top_k}':ndcg,f'recall@{top_k}':recall,f'hr@{top_k}':hr,
                        'n_judged_in_topk':int(sum(movie in truth for movie in recommendation_ids)),
                        'n_unjudged_in_topk':int(sum(movie not in truth for movie in recommendation_ids))})
    return pd.DataFrame(out)


def summarize_metrics(per_scenario, top_k=K_METRICS):
    """Prima media per utente (repliche correlate), poi macro-media fra utenti."""
    stats=[f'ndcg@{top_k}',f'recall@{top_k}',f'hr@{top_k}']
    enriched=per_scenario.merge(scenario_levels[['scenario_id','split']].drop_duplicates(),
                                on='scenario_id',how='left',validate='many_to_one')
    # Nella stessa combinazione utente/modello/level/metric aggrega le repliche per individuo.
    owner=scenarios[['scenario_id','userId']]
    enriched=enriched.merge(owner,on='scenario_id',validate='many_to_one')
    per_user=(enriched.groupby(['split','userId','model','scheme','k'],as_index=False)[stats].mean())
    macro=per_user.groupby(['split','model','scheme','k'],as_index=False)[stats].mean()
    macro['n_users_defined_ndcg']=(per_user.dropna(subset=[stats[0]]).groupby(['split','model','scheme','k']).size()
                .reindex(pd.MultiIndex.from_frame(macro[['split','model','scheme','k']]),fill_value=0).to_numpy())
    return per_user,macro

print('Valutatore A/B pronto. Nessuna valutazione vera finché non si carica un ranking reale.')

Valutatore A/B pronto. Nessuna valutazione vera finché non si carica un ranking reale.


## 7. Auto-test del valutatore (dati fittizi, nessuna MF/GTE)

Costruiamo una top-10 artificiale per un solo scenario, inserendo in posizioni note:

- un film con rating positivo nei **20 target fissi**;
- un film positivo nell'**holdout extra o privato ancora nascosto**, se presente;
- altri film del catalogo non rivelati.

Verifichiamo che B riconosca almeno gli stessi positivi di A e che il valutatore respinga un item già rivelato. **I risultati qui visualizzati NON rappresentano performance sperimentali.**

In [12]:
demo=scenario_levels.loc[(scenario_levels.split=='validation')&(scenario_levels.k==3)].iloc[0]
sid=str(demo.scenario_id);k=int(demo.k)
fixed_gt=evaluation_truth(sid,k,'A');all_hidden=evaluation_truth(sid,k,'B')
positive_fixed=[m for m,r in fixed_gt.items() if r>=POS_THRESHOLD]
positive_extra=[m for m,r in all_hidden.items() if r>=POS_THRESHOLD and m not in fixed_gt]
revealed=observed_lookup.get((sid,k),set())
examples=[]
if positive_fixed:examples.append(positive_fixed[0])
if positive_extra:examples.append(positive_extra[0])
examples.extend([int(m) for m in catalog.movieId if int(m) not in set(examples)|revealed][:K_METRICS-len(examples)])
assert len(examples)==K_METRICS and len(set(examples))==K_METRICS
fake=pd.DataFrame({'scenario_id':sid,'k':k,'model':'DEMO_NON_MODELL0',
                   'movieId':examples,'rank':np.arange(1,K_METRICS+1)})
results_demo=evaluate_rankings(fake)
assert len(results_demo)==2
assert results_demo.loc[results_demo.scheme=='B','n_relevant'].iloc[0] >= \
       results_demo.loc[results_demo.scheme=='A','n_relevant'].iloc[0]
assert set(private_fixed.loc[private_fixed.scenario_id==sid,'movieId']) & revealed == set()
if revealed:
    broken=fake.copy()
    broken.loc[0,'movieId']=next(iter(revealed))
    try:
        evaluate_rankings(broken)
    except ValueError as exc:
        assert 'GIA rivelato' in str(exc)
    else:
        raise AssertionError('Valutatore non ha bloccato un film già rivelato')
print('Auto-test OK: metriche A/B, target extra, filtro anti-leakage.')
display(results_demo)

Auto-test OK: metriche A/B, target extra, filtro anti-leakage.


,scenario_id,k,model,scheme,n_relevant,n_hits_topk,ndcg@10,recall@10,hr@10,n_judged_in_topk,n_unjudged_in_topk
0,validation_u5_r0,3,DEMO_NON_MODELL0,A,13,3,0.368856,0.230769,1.0,4,6
1,validation_u5_r0,3,DEMO_NON_MODELL0,B,22,4,0.507718,0.181818,1.0,5,5


## 8. Valutazione di ranking **reali**, quando disponibili

In futuro, MF/GTE produrranno un file CSV con `scenario_id,k,model,movieId,rank,score`. Questo blocco lo leggerà **solo se esiste**, senza alcuna chiamata a MCP/LLM o ricostruzione del training. Lo stesso ranking è automaticamente valutato in A e B.

Nota: se un modello non produce ranking per certi scenari, le medie devono essere confrontate su un insieme comune di scenari. Il riepilogo qui sotto non sostituisce una verifica di copertura MF/GTE e analisi della significatività.

In [13]:
RANKINGS_PATH = OUT/'predictions'/'ranking_results.csv'
if RANKINGS_PATH.is_file():
    predictions=pd.read_csv(RANKINGS_PATH)
    results=evaluate_rankings(predictions)
    per_user_metrics, macro_metrics=summarize_metrics(results)
    if SAVE_OUTPUTS:
        evaluation_dir=OUT/'evaluations'
        evaluation_dir.mkdir(parents=True,exist_ok=True)
        results.to_csv(evaluation_dir/'metrics_per_scenario_A_B.csv',index=False)
        per_user_metrics.to_csv(evaluation_dir/'metrics_per_user_A_B.csv',index=False)
        macro_metrics.to_csv(evaluation_dir/'metrics_macro_A_B.csv',index=False)
    display(macro_metrics)
else:
    print('Nessun ranking reale ancora disponibile:',RANKINGS_PATH)
    print('Nessun modello è stato eseguito. Il protocollo e il valutatore sono pronti.')

Nessun ranking reale ancora disponibile: /Users/danieleraveglia/Documents/uniMi/TESI/progetto/datasets/processed/recommendation_protocol/rs_coldstart_v1/predictions/ranking_results.csv
Nessun modello è stato eseguito. Il protocollo e il valutatore sono pronti.


## 9. Passi successivi e decisioni documentate

- **Ora:** verificare numero di utenti idonei, copertura training, frequenza di target positivi e salvare artefatti protocollari. Non selezionare repliche in base a metriche del futuro test.
- **Prossimo notebook:** implementare **Most Popular** (fallback k=0) e un primo **Matrix Factorization explicit-feedback con fold-in** per utenti non visti. Leggere esclusivamente `inputs/train_ratings.csv` e le `inputs/visible_interactions.csv` del livello desiderato; non aprire `private_evaluation` dal codice del modello.
- **Confronto contenutistico:** costruire GTE diretto sugli embedding degli item graditi/non graditi, evitando feature MovieLens che contengano rating/tag degli utenti di test.
- **Valutazione:** i modelli scriveranno gli stessi CSV di ranking. Il valutatore calcolerà A e B sugli stessi risultati, **senza ricomputare i ranking**.
- **Dopo:** confronto LLM+GTE e poi esperimento separato pipeline fissa vs orchestrazione condizionale. L'orchestratore aggiornato non viene importato o modificato qui.

**Punti aperti da fissare prima del benchmark definitivo:** split multi-fold vs uno solo, numero di repliche (ora 3), metrica aggiuntiva di penalizzazione dei negativi e trattamento dei bias dovuti ai film mai valutati. Sul set di test non scegliere iperparametri o modalità di valutazione basandosi sul risultato.